<a href="https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The Rule: If position <= 10, Score = impressions * (0.05 - ctr).

Reason Code: high_volume_missed_clicks

Action Label: rewrite_meta_tags

In [ ]:
import os
import pandas as pd

# Clone repo if not already present
if not os.path.exists('/content/flyrankweek1assignment'):
    !git clone https://github.com/ayush13007/flyrankweek1assignment.git /content/flyrankweek1assignment

# Set working directory to repo root
%cd /content/flyrankweek1assignment

# Load data
file_path = 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(file_path)
print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns.")

/content/flyrankweek1assignment
Dataset successfully loaded: 30000 rows, 44 columns.


In [ ]:
import numpy as np
import pandas as pd

# Mapping actual column names from the dataset
pos_col = 'avg_position'
imp_col = 'impressions_last_30d'

# Signal 1: Position vs CTR Bucket Table
if pos_col in df.columns:
    df['position_bucket'] = pd.cut(df[pos_col], bins=[0, 3, 10, 20, 100], labels=['1-3', '4-10', '11-20', '21+'])
    ctr_bucket = df.groupby('position_bucket', observed=False).agg(
        avg_ctr=('ctr', 'mean'),
        median_impressions=(imp_col, 'median'),
        n=('ctr', 'count')
    ).reset_index()

    print("--- Signal 1: Position vs CTR ---")
    print(ctr_bucket)

# Signal 2: Impression Volume Buckets
if imp_col in df.columns:
    df['volume_bucket'] = pd.qcut(df[imp_col], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])
    vol_bucket = df.groupby('volume_bucket', observed=False).agg(
        avg_ctr=('ctr', 'mean'),
        avg_impressions=(imp_col, 'mean'),
        n=('ctr', 'count')
    ).reset_index()

    print("\n--- Signal 2: Impression Volume Buckets ---")
    print(vol_bucket)

--- Signal 1: Position vs CTR ---
  position_bucket   avg_ctr  median_impressions      n
0             1-3  2.714303                 4.0   1141
1            4-10  0.651045               198.0  11842
2           11-20  0.323443               174.0   7273
3             21+  0.211705               148.0   8524

--- Signal 2: Impression Volume Buckets ---
  volume_bucket   avg_ctr  avg_impressions     n
0           Low  1.179536         2.244267  7631
1       Med-Low  0.298420        57.905734  7394
2      Med-High  0.222186       363.451410  7481
3          High  0.327231      5298.572325  7494


## Getting your data into Colab

There are several ways to get your data into the Colab environment. Choose the method that best suits your needs.

### Option 1: Upload directly from your local machine

This method is suitable for smaller files and is temporary, as files uploaded this way will be deleted when your Colab runtime resets. Use the following code to upload your file. After uploading, you will need to create the `data/raw` directory and move your file into it, or adjust the path in the `pd.read_csv` call.

In [ ]:
# Signal Check 1: CTR distribution by Position Buckets
# Updated 'position' to 'avg_position' and 'impressions' to 'impressions_last_30d'
df['position_bucket'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, 100], labels=['1-3', '4-10', '11-20', '21+'])
ctr_bucket = df.groupby('position_bucket', observed=False).agg(
    avg_ctr=('ctr', 'mean'),
    median_impressions=('impressions_last_30d', 'median'),
    n=('ctr', 'count')
).reset_index()

print("--- Signal 1: Position vs CTR ---")
print(ctr_bucket)

# Signal Check 2: Impression Volume vs CTR Opportunity
df['volume_bucket'] = pd.qcut(df['impressions_last_30d'], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])
vol_bucket = df.groupby('volume_bucket', observed=False).agg(
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('impressions_last_30d', 'mean'),
    n=('ctr', 'count')
).reset_index()

print("\n--- Signal 2: Impression Volume Buckets ---")
print(vol_bucket)

--- Signal 1: Position vs CTR ---
  position_bucket   avg_ctr  median_impressions      n
0             1-3  2.714303                 4.0   1141
1            4-10  0.651045               198.0  11842
2           11-20  0.323443               174.0   7273
3             21+  0.211705               148.0   8524

--- Signal 2: Impression Volume Buckets ---
  volume_bucket   avg_ctr  avg_impressions     n
0           Low  1.179536         2.244267  7631
1       Med-Low  0.298420        57.905734  7394
2      Med-High  0.222186       363.451410  7481
3          High  0.327231      5298.572325  7494


### Option 2: Mount Google Drive

Mounting Google Drive allows you to access files stored in your Drive directly from Colab. This is a more permanent solution as your files persist across sessions. You'll need to authorize Colab to access your Google Drive.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Change directory to your cloned repo root
%cd /content/flyrankweek1assignment

# 2. Load dataset directly from local repo workspace
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 3. Column mapping based on dataset schema
pos_col = 'avg_position' if 'avg_position' in df.columns else 'position'
imp_col = 'impressions_last_30d' if 'impressions_last_30d' in df.columns else 'impressions'

# 4. Apply baseline rule logic
df['action_label'] = np.where((df[pos_col] <= 10) & (df['ctr'] < 0.05), 'rewrite_meta_tags', 'no_action')
df['reason_code'] = np.where(df['action_label'] == 'rewrite_meta_tags', 'high_volume_missed_clicks', 'none')
df['baseline_score'] = np.where(
    df['action_label'] == 'rewrite_meta_tags',
    df[imp_col] * (0.05 - df['ctr']),
    0.0
)

# 5. Sort and rank
ranked_queue = df[df['action_label'] == 'rewrite_meta_tags'].sort_values(by='baseline_score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

# 6. Select core output columns
output_cols = ['rank', 'baseline_score', 'action_label', 'reason_code', pos_col, 'ctr', imp_col]
if 'content_id' in df.columns:
    output_cols.insert(1, 'content_id')
elif 'content_hash_id' in df.columns:
    output_cols.insert(1, 'content_hash_id')

# 7. Export to CSV
os.makedirs('work/outputs', exist_ok=True)
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Success! Ranked queue written to work/outputs/baseline_action_score.csv with {len(ranked_queue)} actionable items.")

/content/flyrankweek1assignment
Success! Ranked queue written to work/outputs/baseline_action_score.csv with 6083 actionable items.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# --- Section 3: Inspect Actual Top 10 ---
print("=== TOP 10 RANKED QUEUE ITEMS ===")
print(ranked_queue[output_cols].head(10).to_string(index=False))

# --- Section 4: Weak Picks & Data Leakage Check ---
print("\n=== SECTION 4 AUDIT ===")
# Identify weak picks (position > 8 with lower relative impression volume)
weak_picks = ranked_queue[(ranked_queue[pos_col] > 8) & (ranked_queue[imp_col] < ranked_queue[imp_col].quantile(0.5))]
print(f"Weak picks identified for review: {len(weak_picks)}")

# Verify no future-window or target features were used in ranking logic
used_cols = [pos_col, imp_col, 'ctr']
leakage_found = any(any(term in col.lower() for term in ['future', 'next', 'target', 'label', 'conversion']) for col in used_cols)
print(f"Data Leakage Check Passed: {not leakage_found}")

=== TOP 10 RANKED QUEUE ITEMS ===
 rank           content_id  baseline_score      action_label               reason_code  avg_position  ctr  impressions_last_30d
    1 content_4a6607efcb46         4892.12 rewrite_meta_tags high_volume_missed_clicks           2.2 0.01                122303
    2 content_8451fc6f034d         3379.16 rewrite_meta_tags high_volume_missed_clicks           2.3 0.03                168958
    3 content_c8e9d6ab9013         3166.30 rewrite_meta_tags high_volume_missed_clicks           9.7 0.00                 63326
    4 content_c84a0ab98e90         1986.74 rewrite_meta_tags high_volume_missed_clicks           7.8 0.03                 99337
    5 content_39881853ef0c         1507.08 rewrite_meta_tags high_volume_missed_clicks           7.2 0.01                 37677
    6 content_b115f7c74779         1022.30 rewrite_meta_tags high_volume_missed_clicks           8.0 0.03                 51115
    7 content_d274ac4158ef          798.76 rewrite_meta_tags high_volu

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# 1. Programmatically identify weak picks (Position 8-10 with bottom 50% impressions)
actionable_mask = ranked_queue['action_label'] == 'rewrite_meta_tags'
p8_10_mask = ranked_queue[pos_col] >= 8
low_vol_mask = ranked_queue[imp_col] < ranked_queue[imp_col].median()

weak_picks = ranked_queue[p8_10_mask & low_vol_mask]
print(f"--- WEAK PICKS IDENTIFIED ---")
print(f"Total weak picks found: {len(weak_picks)} (Positions 8-10 with below-median impressions)")
if len(weak_picks) > 0:
    print(weak_picks[['rank', pos_col, imp_col, 'ctr', 'baseline_score']].head())

# 2. Programmatic Leakage Audit
scanned_cols = [pos_col, imp_col, 'ctr']
leakage_keywords = ['future', 'next', 'target', 'label', 'flag', 'converted', 't+1']

leakage_found = [
    col for col in scanned_cols
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

print("\n--- LEAKAGE AUDIT VERDICT ---")
if not leakage_found:
    print("VERDICT: PASSED (Clean baseline score using only past/current observation window features).")
else:
    print(f"VERDICT: FAILED (Leakage columns detected: {leakage_found})")

--- WEAK PICKS IDENTIFIED ---
Total weak picks found: 398 (Positions 8-10 with below-median impressions)
      rank  avg_position  impressions_last_30d  ctr  baseline_score
3349  3350           8.0                     1  0.0            0.05
3373  3374           8.3                     1  0.0            0.05
3392  3393           8.0                     1  0.0            0.05
3469  3470           8.4                     1  0.0            0.05
3502  3503           9.0                     1  0.0            0.05

--- LEAKAGE AUDIT VERDICT ---
VERDICT: PASSED (Clean baseline score using only past/current observation window features).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.